In [14]:
from dataPrep import DatasetPreparation
from model import ConvModel
import tensorflow as tf
import cv2


In [15]:
vgg_backbone = tf.keras.applications.vgg16.VGG16(
    include_top= False,
    weights= 'imagenet',
    input_shape=(256,256,3) 
)

In [16]:
vgg_backbone.summary()

Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 256, 256, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 256, 256, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 128, 128, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 128, 128, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 128, 128, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 64, 64, 128)       0     

In [17]:
feature_map = [layer.output for layer in vgg_backbone.layers[1:]]
feature_map_model = tf.keras.Model(
    inputs = vgg_backbone.input,
    outputs = feature_map
)
feature_map_model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 256, 256, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 256, 256, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 128, 128, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 128, 128, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 128, 128, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 64, 64, 128)       0   

In [18]:

train_path = r"D:\RECURSOS DE TRABAJO\Base de Datos para IA\Emotions Dataset\Emotions Dataset\train"

CONFIGURATION = {
    "IM_SIZE":256,
    "CLASS_NAMES": ["angry","happy","sad"],
    "BATCH_SIZE":32,
    "SEED":123,
}

prep = DatasetPreparation()
train, val, test = prep.load_all(train_path, CONFIGURATION)

Found 6799 files belonging to 3 classes.
Using 5440 files for training.
Found 6799 files belonging to 3 classes.
Using 1359 files for validation.
Found 2278 files belonging to 3 classes.


In [ ]:
import cv2
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Ruta completa de la imagen
ruta_imagen = r'D:\RECURSOS DE TRABAJO\Base de Datos para IA\Emotions Dataset\Emotions Dataset\train\angry\8853.jpg_brightness_2.jpg'

test_image = cv2.imread(ruta_imagen)

if test_image is None:
    print("No se encontró la imagen. Verifica la ruta y el nombre del archivo.")
else:
    SIZE = (224, 224)
    test_image = cv2.resize(test_image, SIZE)
    im = tf.convert_to_tensor(test_image, dtype=tf.float32)
    im = im / 255.0
    img_array = tf.expand_dims(im, axis=0)
    print(img_array.shape)

    # Cargar el modelo (ajusta la ruta a tu modelo)
    model = tf.keras.models.load_model('C:/Users/vpn/Documents/GitHub/Ciencia-de-datos/Deteccion_de_Expresiones_Humanas/modelo.h5')

    # Método Grad-CAM
    def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
        grad_model = tf.keras.models.Model(
            [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
        )
        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(img_array)
            if pred_index is None:
                pred_index = tf.argmax(predictions[0])
            class_channel = predictions[:, pred_index]
        grads = tape.gradient(class_channel, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)
        heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
        return heatmap.numpy()

    # Cambia 'last_conv_layer_name' por el nombre de la última capa convolucional de tu modelo
    last_conv_layer_name = 'conv2d'  # Ajusta según tu modelo

    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)

    # Visualización
    plt.matshow(heatmap)
    plt.title("Grad-CAM Heatmap")
    plt.show()

(1, 224, 224, 3)


OSError: No file or directory found at ruta_a_tu_modelo.h5